# Model Quality Gate for a Cancer-Diagnosis Classifier: Validation, Metric Blind Spots & Drift Detection

*Practice notebook — 2026-11-23 — topic: ML testing and MLOps*

**Problem statement.** You're the ML quality engineer at a hospital analytics team. A colleague has
trained a logistic regression model on the Wisconsin Breast Cancer diagnostic dataset
(`sklearn.datasets.load_breast_cancer` — real biopsy measurements, binary malignant/benign label) to
flag malignant tumors from cell-nuclei measurements. Before any model version ships, it must pass an
automated **quality gate**: a reusable suite of checks, not a one-off "looks fine" glance at a single
metric. You're going to build that gate.

The gate has three layers, each targeting a different way ML systems fail silently in production:

1. **Data validation** — does an incoming batch of patient data even look like what the model was
   trained on (right columns, right types, no nulls, physically plausible values)?
2. **Metric evaluation beyond one aggregate number** — does the model clear a bar not just on overall
   accuracy, but on the metric that actually matters here (recall on the *malignant* class — a false
   negative is a missed cancer), and is performance consistent across patient segments, or does a good
   overall score hide a segment where it quietly fails?
3. **Drift detection** — has a new batch of incoming data statistically drifted away from the
   distribution the model was trained and validated on? Silent input drift is one of the most common
   real-world causes of ML quality degradation, and it's invisible if you only ever look at training-time
   metrics.

**What you should produce**, concretely:

- `validate_schema(df, schema)` — raises a single `AssertionError` listing *every* violation found
  (columns, dtypes, nulls, physically-implausible ranges), not just the first one it hits.
- `evaluate_classifier(model, X, y, segment_col=None, segment_bins=4)` — returns overall accuracy plus
  precision/recall/F1 **for the malignant class specifically**, and, when a segment column is given,
  the same metrics computed per segment (e.g. by tumor-size quartile).
- `check_quality_gate(metrics, min_accuracy, min_malignant_recall, max_segment_recall_gap)` — returns
  `(passed, reasons)`, failing when overall accuracy looks fine but malignant recall doesn't, or when
  the recall gap between the best and worst segment is too large.
- `compute_psi(reference, current, buckets)` and `detect_drift(reference_df, current_df, feature_cols,
  psi_threshold, ks_alpha)` — a per-feature drift report combining the Population Stability Index and a
  Kolmogorov–Smirnov test.
- `run_quality_gate(...)` — wires all three layers into one pass/fail decision with a structured report,
  and you'll run it against: (a) a clean held-out split (should **PASS**), (b) a model that's been
  biased to quietly under-recall the malignant class (should **FAIL** the metric gate even though its
  accuracy alone looks acceptable), and (c) a batch with injected distribution shift (should **FAIL**
  the drift gate).

We inject *synthetic* drift for part (c) on purpose: to test a drift detector you need a batch where you
*know* the ground truth (drifted / not drifted), which real future data can never promise you while
you're still writing the test.

## Starter cells

Imports, data loading, train/test split, and a baseline model are provided. The scaffolded functions
below have signatures and TODOs only — no solution logic. Fill them in, then run the demo cells at the
bottom to see your gate PASS on clean data and FAIL on the two broken scenarios.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
MALIGNANT = 0  # sklearn's encoding: 0 = malignant, 1 = benign

data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={"target": "diagnosis"})
feature_cols = [c for c in df.columns if c != "diagnosis"]

print(df.shape)
print(data.target_names)  # ['malignant' 'benign']
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols], df["diagnosis"],
    test_size=0.25, random_state=RANDOM_STATE, stratify=df["diagnosis"],
)

baseline_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
baseline_model.fit(X_train, y_train)

print("train:", X_train.shape, " test:", X_test.shape)

In [ ]:
def build_reference_schema(X_train, feature_cols):
    """TODO: build and return a reference schema dict from the training data.

    Should contain at least:
      - "columns": the expected column list/order
      - "dtypes": {column -> dtype as a string}
      - "ranges": {column -> (lo, hi)} -- a PHYSICALLY PLAUSIBLE range, not the exact
        train min/max (that's too brittle: a real test batch will naturally have a few
        values just outside the training sample's observed min/max). Think about what
        bound actually makes sense for cell-measurement features.
    """
    # TODO
    raise NotImplementedError


def validate_schema(df_check, schema):
    """TODO: validate df_check against `schema`.

    Collect EVERY violation (don't return on the first one) covering:
      - column list/order matches schema["columns"]
      - dtype of each column matches schema["dtypes"]
      - no nulls in any column
      - no values outside schema["ranges"][column]

    Raise a single AssertionError listing all violations if any are found.
    Return True if validation passes.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def evaluate_classifier(model, X, y, segment_col=None, segment_bins=4):
    """TODO: evaluate `model` on (X, y).

    Return a dict:
      {
        "overall": {"accuracy": ..., "malignant_precision": ..., "malignant_recall": ..., "malignant_f1": ...},
        "segments": {segment_label: {"n": ..., "accuracy": ..., "malignant_recall": ...}, ...}  # only if segment_col given
      }

    For segments, bucket X[segment_col] into `segment_bins` quantile bins (pd.qcut) and compute
    the same per-segment metrics. Skip any segment with too few rows or only one class present.
    Remember: malignant is the POSITIVE class of interest here (pos_label=MALIGNANT).
    """
    # TODO
    raise NotImplementedError


def check_quality_gate(metrics, min_accuracy=0.90, min_malignant_recall=0.90, max_segment_recall_gap=0.20):
    """TODO: decide pass/fail from the metrics dict `evaluate_classifier` returned.

    Return (passed: bool, reasons: list[str]).
    Fail (with a reason) if:
      - overall accuracy < min_accuracy
      - overall malignant recall < min_malignant_recall   <-- this is the one that catches the
        "accuracy looked fine" trap
      - if segments are present, (max segment malignant_recall - min segment malignant_recall) > max_segment_recall_gap
    """
    # TODO
    raise NotImplementedError

In [ ]:
def compute_psi(reference, current, buckets=10):
    """TODO: compute the Population Stability Index between two 1-D numeric arrays.

    Standard recipe:
      1. Build `buckets` quantile bin edges from `reference`.
      2. Compute the % of reference and current values falling in each bin.
      3. PSI = sum((cur_pct - ref_pct) * ln(cur_pct / ref_pct)) over bins.
      4. Guard against zero-division / log(0) for empty bins (clip to a tiny epsilon).

    Return a float. As a rule of thumb: PSI < 0.1 ~ no meaningful shift, 0.1-0.2 ~ moderate,
    > 0.2 ~ significant shift.
    """
    # TODO
    raise NotImplementedError


def detect_drift(reference_df, current_df, feature_cols, psi_threshold=0.2, ks_alpha=0.05):
    """TODO: build a per-feature drift report.

    For each column in feature_cols, compute PSI (via compute_psi) and a two-sample
    Kolmogorov-Smirnov test (scipy.stats.ks_2samp) between reference_df[col] and current_df[col].
    Flag a column as drifted if PSI > psi_threshold OR the KS test's p-value < ks_alpha.

    Return:
      {
        "features": {col: {"psi": ..., "ks_p": ..., "drifted": bool}, ...},
        "n_drifted": int,
        "overall_drifted": bool,  # True if n_drifted > 0
      }
    """
    # TODO
    raise NotImplementedError

In [ ]:
def run_quality_gate(model, X_ref, y_ref, X_new, y_new, feature_cols, schema, segment_col=None):
    """TODO: orchestrate the full gate and return (overall_pass: bool, report: dict).

    report should capture, at minimum:
      - "schema_ok": bool (+ "schema_error" message if it failed)
      - "metrics": the evaluate_classifier(...) output on (X_new, y_new)
      - "metric_gate_passed": bool, "metric_reasons": list[str]
      - "drift": the detect_drift(...) output comparing X_ref vs X_new

    overall_pass should be True only if schema is ok AND the metric gate passed AND no drift
    was detected. Don't let an exception from validate_schema crash the whole function --
    catch it and record the failure in the report instead.
    """
    # TODO
    raise NotImplementedError

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Why a plausibility range, not the training min/max?** A tight `(train_min, train_max)` range check
will almost always fire on a fresh, perfectly healthy test split, just from ordinary sampling variation
— that's a brittle test, not a useful one. Here the range check uses a generous physical-plausibility
bound (`0` to `1.5 x training max`, since every feature in this dataset is a non-negative cell
measurement) to catch gross anomalies (corrupted rows, unit errors, sensor faults), while drift
detection (PSI/KS) is the right tool for the subtler "distribution shifted but values are still
individually plausible" case.

```python
def build_reference_schema(X_train, feature_cols):
    return {
        "columns": feature_cols,
        "dtypes": {c: str(X_train[c].dtype) for c in feature_cols},
        # physical plausibility buffer, not a tight train min/max (too brittle -- a clean
        # future batch will naturally have a few points just outside the observed range)
        "ranges": {c: (0.0, float(X_train[c].max()) * 1.5) for c in feature_cols},
    }


def validate_schema(df_check, schema):
    errors = []
    if list(df_check.columns) != schema["columns"]:
        errors.append(f"column mismatch: expected {schema['columns']}, got {list(df_check.columns)}")
    else:
        for c in schema["columns"]:
            if str(df_check[c].dtype) != schema["dtypes"][c]:
                errors.append(f"{c}: dtype mismatch, expected {schema['dtypes'][c]}, got {df_check[c].dtype}")
            if df_check[c].isnull().any():
                errors.append(f"{c}: contains nulls")
            lo, hi = schema["ranges"][c]
            below = df_check[c] < lo
            above = df_check[c] > hi
            if below.any():
                errors.append(f"{c}: {below.sum()} values below plausible min {lo:.4f}")
            if above.any():
                errors.append(f"{c}: {above.sum()} values above plausible max {hi:.4f}")
    if errors:
        raise AssertionError("Schema validation failed:\n" + "\n".join(errors))
    return True
```

**Why malignant recall, not just accuracy?** This dataset is only mildly imbalanced (37% malignant), so
accuracy alone isn't *as* misleading here as it would be on a 99/1 fraud-style dataset -- but it's still
possible to bias a model (e.g. via `class_weight`) toward the majority class in a way that keeps overall
accuracy comfortably above a naive threshold while the recall on the class that actually matters quietly
degrades. A quality gate for a safety-critical classifier has to check the metric that matches the cost
of the error, not whichever metric happens to look best.

```python
def evaluate_classifier(model, X, y, segment_col=None, segment_bins=4):
    preds = model.predict(X)
    overall = {
        "accuracy": accuracy_score(y, preds),
        "malignant_precision": precision_score(y, preds, pos_label=MALIGNANT),
        "malignant_recall": recall_score(y, preds, pos_label=MALIGNANT),
        "malignant_f1": f1_score(y, preds, pos_label=MALIGNANT),
    }
    result = {"overall": overall}
    if segment_col is not None:
        segments = pd.qcut(X[segment_col], q=segment_bins, duplicates="drop")
        seg_metrics = {}
        for seg in segments.unique():
            mask = (segments == seg).to_numpy()
            if mask.sum() < 5 or y[mask].nunique() < 2:
                continue
            seg_metrics[str(seg)] = {
                "n": int(mask.sum()),
                "accuracy": accuracy_score(y[mask], preds[mask]),
                "malignant_recall": recall_score(y[mask], preds[mask], pos_label=MALIGNANT, zero_division=0),
            }
        result["segments"] = seg_metrics
    return result


def check_quality_gate(metrics, min_accuracy=0.90, min_malignant_recall=0.90, max_segment_recall_gap=0.20):
    reasons = []
    overall = metrics["overall"]
    if overall["accuracy"] < min_accuracy:
        reasons.append(f"overall accuracy {overall['accuracy']:.3f} < {min_accuracy}")
    if overall["malignant_recall"] < min_malignant_recall:
        reasons.append(f"malignant recall {overall['malignant_recall']:.3f} < {min_malignant_recall}")
    if metrics.get("segments"):
        recalls = [m["malignant_recall"] for m in metrics["segments"].values()]
        gap = max(recalls) - min(recalls)
        if gap > max_segment_recall_gap:
            reasons.append(f"segment malignant-recall gap {gap:.3f} > {max_segment_recall_gap}")
    return (len(reasons) == 0), reasons
```

**PSI + KS together:** PSI is a magnitude-of-shift summary that's easy to threshold and communicate;
KS is a proper statistical test for "do these two samples come from the same distribution." Using both
catches shifts that are statistically significant but numerically small (KS-sensitive) as well as shifts
that are large in magnitude but might not hit classical significance on a small batch (PSI-sensitive).

```python
def compute_psi(reference, current, buckets=10):
    breakpoints = np.quantile(reference, np.linspace(0, 1, buckets + 1))
    breakpoints[0] -= 1e-9
    breakpoints[-1] += 1e-9
    ref_counts, _ = np.histogram(reference, bins=breakpoints)
    cur_counts, _ = np.histogram(current, bins=breakpoints)
    ref_pct = np.where(ref_counts == 0, 1e-6, ref_counts) / len(reference)
    cur_pct = np.where(cur_counts == 0, 1e-6, cur_counts) / len(current)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def detect_drift(reference_df, current_df, feature_cols, psi_threshold=0.2, ks_alpha=0.05):
    report = {}
    for c in feature_cols:
        psi = compute_psi(reference_df[c].to_numpy(), current_df[c].to_numpy())
        ks_stat, ks_p = ks_2samp(reference_df[c], current_df[c])
        drifted = (psi > psi_threshold) or (ks_p < ks_alpha)
        report[c] = {"psi": psi, "ks_p": float(ks_p), "drifted": bool(drifted)}
    n_drifted = sum(1 for r in report.values() if r["drifted"])
    return {"features": report, "n_drifted": n_drifted, "overall_drifted": n_drifted > 0}


def run_quality_gate(model, X_ref, y_ref, X_new, y_new, feature_cols, schema, segment_col=None):
    report = {}
    try:
        validate_schema(X_new, schema)
        report["schema_ok"] = True
    except AssertionError as e:
        report["schema_ok"] = False
        report["schema_error"] = str(e)

    metrics = evaluate_classifier(model, X_new, y_new, segment_col=segment_col)
    gate_passed, reasons = check_quality_gate(metrics)
    report["metrics"] = metrics
    report["metric_gate_passed"] = gate_passed
    report["metric_reasons"] = reasons

    drift = detect_drift(X_ref, X_new, feature_cols)
    report["drift"] = drift

    report["overall_pass"] = report["schema_ok"] and gate_passed and not drift["overall_drifted"]
    return report["overall_pass"], report
```

**Running all three scenarios** (clean baseline, an accuracy-paradox model, and an injected-drift batch):

```python
reference_schema = build_reference_schema(X_train, feature_cols)

# (a) clean baseline model + clean held-out split -> should PASS
ok, report = run_quality_gate(
    baseline_model, X_train, y_train, X_test, y_test, feature_cols, reference_schema, segment_col="mean radius"
)
print("baseline:", ok, report["metric_gate_passed"], report["drift"]["overall_drifted"])

# (b) a model biased to under-recall the malignant class -- overall accuracy still looks fine
biased_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight={MALIGNANT: 0.05, 1: 1.0})
)
biased_model.fit(X_train, y_train)
ok_b, report_b = run_quality_gate(
    biased_model, X_train, y_train, X_test, y_test, feature_cols, reference_schema, segment_col="mean radius"
)
print("biased model:", ok_b, report_b["metric_gate_passed"], report_b["metric_reasons"])
# -> accuracy ~0.94 (looks fine!) but malignant recall ~0.83 trips the gate

# (c) inject synthetic drift (simulating a sensor/calibration shift) into an otherwise-clean batch
rng = np.random.default_rng(RANDOM_STATE)
X_shifted = X_test.copy()
X_shifted["mean radius"] = X_shifted["mean radius"] * 1.25 + 2.0
X_shifted["mean texture"] = X_shifted["mean texture"] + rng.normal(5, 1, size=len(X_shifted))
ok_c, report_c = run_quality_gate(
    baseline_model, X_train, y_train, X_shifted, y_test, feature_cols, reference_schema, segment_col="mean radius"
)
print("shifted batch:", ok_c, report_c["drift"]["overall_drifted"], report_c["drift"]["n_drifted"])
# -> metric gate still passes (model wasn't retrained on bad data), but drift gate correctly fails
```

Expected output pattern: `baseline -> PASS`, `biased model -> FAIL (metric gate, malignant recall)`,
`shifted batch -> FAIL (drift gate)`. Each failure mode is caught by the specific layer designed to
catch it -- that separation (schema vs. metric vs. drift) is exactly what makes the gate debuggable in
a real CI pipeline: a failing run tells you *which* kind of problem to go look at.

</details>